In [ ]:
import asyncio
from asyncio import CancelledError, Future

from utils import async_timed, delay

In [ ]:
async def main():
    long_task = asyncio.create_task(delay(10))

    seconds_elapsed = 0

    while not long_task.done():
        print(f"Task not finished, checking again in a second.")
        await asyncio.sleep(1)
        seconds_elapsed += 1
        if seconds_elapsed == 5:
            long_task.cancel()
    try:
        await long_task
    except CancelledError:
        print(f"Out task was cancelled.")


await main()

In [ ]:
# 取消任务
async def main():
    delay_task = asyncio.create_task(delay(10))
    try:
        result = await asyncio.wait_for(delay_task, timeout=5)
        print(result)
    except asyncio.exceptions.TimeoutError:
        print(f"Got a timeout.")
        print(f"Was the task cancelled？ {delay_task.cancelled()}")


await main()

In [ ]:
# 保护任务避免取消，但有超时提示
async def main():
    task = asyncio.create_task(delay(10))
    try:
        result = await asyncio.wait_for(asyncio.shield(task), timeout=5)
        print(result)
    except asyncio.exceptions.TimeoutError:
        print(f"Task took longer than five seconds, it will finish soon!")
        result = await task
        print(result)


await main()

In [ ]:
# future对象包含一个在未来某个时间点获得但目前可能还不存在的值
my_future = Future()

print(f"Is my_future done? {my_future.done()}")

my_future.set_result(42)

print(f"Is my_future done? {my_future.done()}")
print(f"What is the result of my_future? {my_future.result()}")

In [ ]:
# 等待一个future
async def set_future_value(future: Future) -> None:
    await asyncio.sleep(1)
    future.set_result(42)


def make_request() -> Future:
    future = Future()
    asyncio.create_task(set_future_value(future))
    return future


async def main():
    future = make_request()
    print(f"Is the future done? {future.done()}")
    value = await future
    print(f"Is the future done? {future.done()}")
    print(value)


await main()

In [ ]:
def show_result(result: Future):
    print(result.result())


async def main():
    delay_task = asyncio.create_task(delay(10))
    delay_task.add_done_callback(show_result)
    await delay_task


await main()

In [ ]:
@async_timed()
async def main():
    task_one = asyncio.create_task(delay(2))
    task_two = asyncio.create_task(delay(3))
    await task_one
    await task_two


await main()